# Programming Exercise 1 : Search Algorithms and Uninformed Search

In this notebook you are going to find code and exercises for the first part of your programming project. For this project, you are going to work on a path-finding game, dubbed as RoBoToMaZe, which we have created just for you! 

**What to submit?** Just this notebook, with your code and answers for the various tasks and questions.

## RoBoToMaZe Rules, Instructions and pygame Installation

RoBoToMaZe is a simple variant of a path finding game. Given a two-dimensional grid space (that is, space divided into cells), we have a cell that hosts our token (the robot, if you prefer), a cell that corresponds to the goal (where the robot has to go) and then cells that are obstacles and cells that are free. 

The robot is depicted as a yellow circle, the goal as a red cell, obstacles are in blue and all free cells, where the robot can move to, are in white.

The robot is allowed to move one cell up, down, right or left, assuming that no obstacle is in one of these positions. To move the robot, all you have to do is to click to the cell you want to move the robot to, but only within the robot's legal neighbourhood.

The goal is, to move the robot to the goal (red) cell. It is pretty easy and not an inspiring game, but it serves as a good canvas for designing, developing and testing search algorithms, like the ones we learn in our class.

In the window of the game, you can quit the game, or draw a new game, by clicking the corresponding buttons.

To play the game, you can simply run the next two cells. The first contains all the necessary libraries. The second, a simple implementation for the game. The game is implemented through a series of functions. Some of these functions support graphical parts of the game and others logical parts. The game is essentially run through the main function.

We assume you all have a working python installation on your system (please make sure you do and that you have python version 3.x). You should also install pygame. That is very easy, and can be done by simply executing: 
`pip install pygame`. 

If pip does not work for you, then please just try the method you normally use to install python packages. 

We recommend you have a python version 3.10 and above and a pygame version 2.5.2 and above. All the code and examples given for this course are written and tested with Python 3.12.3 and pygame 2.6.1.

<div>
<img src="attachment:36239cf6-cad3-4482-b764-83025e11ffb9.png" width="500"/>
</div>

In [18]:
## In this cell, we import all the libraries we are going to use for this notebook
## I recommend you adopt this good-programming practise where all necessary
## libraries for your script are imported and found at the top of the script.
# Make sure you can run this cell with no issues.
# That means, your system should have and support all the necessary libraries.

import pygame
from pygame.locals import *
import sys
import numpy as np
from numpy import random
import random
import time

In [19]:
def make_game_basics(res, game_title):
    ''' Function for initializing a resolution for the game window, the game screen, display a name for the game and define colours that
    will be used throughout the graphical parts of the game. 
    Inputs:
    res: the resolution of the screen of the game
    game_title : how the game will be called - that will appear at the top of the game window
    Outputs:
    res : as before
    screen : a window pygame object where everything will be superimposed on while the game is running
    colors : the defined colours to be used throughout the game
    width : the width of the screen window (through the resolution variable)
    height : the height of the screen window (through the resolution variable)'''
    
    res = res 
    screen = pygame.display.set_mode(res) 
    
    pygame.display.set_caption(game_title)

    colors = {"white" :(255,255,255), "black" : (0,0,0), 
              "gray" :(100,100,100), "yellow" : (168, 135, 50), "red": (229, 27, 27),
              "blue" : (33, 27, 229), "pink": (242, 192, 227)}
    
    width = screen.get_width() 
    height = screen.get_height() 

    return res, screen, colors, width, height

def make_text(myfont, text, color):
    '''Function for creating text given a quote and color provided
    inputs: 
    myfont which is the font of text we will dsiplay
    text which is the quote to be displayed
    color which is the color of text to be displayed
    outputs: new_text which is the text with the specific 
    font and color, new_text_size which is the size of the text in pixels
    '''
    new_text = myfont.render(text, True ,color)
    new_text_size = new_text.get_size()
    new_text_rect = new_text.get_rect()
    
    return new_text, new_text_size
#this will create a rectangle with a text inside it with the given parameters we give it. 
def make_rect_with_text(surface, color, c1, c2, w, h, text, text_c1, text_c2):
    pygame.draw.rect(surface, color, [c1, c2, w, h])
    surface.blit(text, (text_c1, text_c2))

def gridCoords(cellSize):
    '''Function for creating a square shaped grid with a given cell size 
    inputs: 
    cellSize which is the size of the desired square cells in pixels
    outputs: 
    cell_cords which is a square grid with the given dimensions from cellSize and the width and height of the screen window
    '''
    cells_coords = []
    for x in range(0, width, cellSize):
        for y in range(0, height-90, cellSize):
            cells_coords.append((x,y))

    return(cells_coords)
    
def drawGrid(surface, color, cells_coords, cellSize):
    '''Function for drawing a grid on the screen
    inputs: 
    surface which is the screen window where the grid will be drawn
    color which is the color of the grid lines
    cells_coords which is the coordinates of the grid cells
    cellSize which is the size of the grid cells in pixels
    outputs:
    rects_coords which is a list of tuples containing the rectangle object and its coordinates for each
    cell in the grid
    rectangles are also drawn on the surface with the given color and cell size
    '''
    rects_coords = []
    for item in cells_coords:
        rect = pygame.Rect(item[0], item[1], cellSize, cellSize)
        pygame.draw.rect(surface, color, rect, 1)
        rects_coords.append((rect, item))

    return(rects_coords)

def drawRobot(surface, color, coords, cellSize):
    '''Function for drawing the robot on the screen
    inputs:
    surface which is the screen where the robot will be drawn
    color which is the color of the robot
    coords which is the coordinates of the robot
    cellSize which is the size of the grid cells in pixels
    outputs:
    None
    rectangles and cirlces that make up the robot are drawn on the surface with the given color and cell size
    '''
    rect = pygame.Rect(coords[0], coords[1], cellSize, cellSize)
    pygame.draw.rect(surface, colors["black"], rect,1)
    
    pygame.draw.circle(surface, color, [coords[0]+(cellSize/2), coords[1]+(cellSize/2)], (cellSize/2)-2)

def drawGoal(surface, color, coords, cellSize):
    
    rect = pygame.Rect(coords[0]+2, coords[1]+2, cellSize-5, cellSize-5)
    pygame.draw.rect(surface,color, rect)

def drawObstacles(surface, color, coords, cellSize):

    for item in coords:
        
        rect = pygame.Rect(item[0], item[1], cellSize, cellSize)
        pygame.draw.rect(surface, color, rect)

def sample_cells(cells, number=1):
    '''Function for sampling cells from the grid to place all the pieces
    inputs:
    cells: a list of tuples with coordinates of the grid cells
    number: the number of cells to sample for obstacles
    outpus:
    robot cell: tuple of coordiantes for the robot cell
    gaol cell: tuple of coordinates for the goal cell
    random.sample(cells, number): a list of tuples with coordinates for the sampled obstacle cells
    '''

    #task 1 part 2a change: the robot must be on the grid, it must not be on the goal cell (600,300) and it must not be on an obstacle cell. 
        #(150,300) is not an obstacle cell since its not on the list of sampled obstacle cells. 
    robot_cell = (150, 300)

    goal_cell = (600,300)
    cells.remove(robot_cell)
    cells.remove(goal_cell)
    
    return robot_cell, goal_cell, random.sample(cells, number)

def legalMove(robot, clicked, rect_coords, obstacles, width, height, cellSize):
    '''Function for checking if the move is legal and returns the new position of the robot
    inputs: 
    robot which is a tuple of coordinates for the robot 
    clicked which is a tuple of coordiantes for the clicked cell
    rect_coords which is a list of tuples with coordinates for the grid cells
    obstacles which is a list of tuples with coordinates for the obstacle cells
    width which is the width of the screen window
    height which is the height of the screen window
    cellSize which is the size of the grid cells in pixels
    outputs:
    new_robot_x, new_robot_y which is a tuple of coordinates for the new position of
    the robot if the move is legal, otherwise it returns the original position
    '''
    new_robot_x, new_robot_y = robot
    
    x,y = clicked
    
    # if the click is within the legal play area
    if 0 < x < width and 0 < y < height-90:
        
        #find the (x,y) of the cell that is clicked:
        x, y = clicked
        for i in range(len(rect_coords)):
            j = rect_coords[i]
            if j[0] <= x <= j[0]+cellSize and j[1] <= y <= j[1]+cellSize:
                cell_x,cell_y = j[0], j[1]
        
        # check if the cell is in the neighbourhood of the robot
        rX,rY = robot
        if (cell_x == rX+cellSize and cell_y == rY) or (cell_x ==rX-cellSize and cell_y == rY) or (cell_x == rX and cell_y ==rY+cellSize) or (cell_x == rX and cell_y == rY-cellSize):
            
            # check if the cell is an obstacle
            if (cell_x,cell_y) not in obstacles:
                
                new_robot_x, new_robot_y = cell_x, cell_y
                
            else:
                new_robot_x, new_robot_y = robot
    return new_robot_x, new_robot_y

def reachedGoal(path, clicked, goal, width, height, cellSize):
    
    flag = False
    x,y = clicked
    
    #check if the goal cell is clicked
    if goal[0]<= x <=goal[0]+cellSize and goal[1] <= y <= goal[1]+cellSize:
        
        #check if the robot had been in the neighbourhood in the previous step
        lastX, lastY = path[-2]
        if (lastX == goal[0]+cellSize and lastY == goal[1]) or (lastX ==goal[0]-cellSize and lastY == goal[1]) or (lastX == goal[0] and lastY ==goal[1]+cellSize) or (lastX == goal[0] and lastY == goal[1]-cellSize):

            flag = True    
                
    return flag

def get_valid_neighbours(current_cell, all_cells, cellSize, obstacles):
    # return the neighbours of the current position of the robot, which are not obstacles
    # even if one of the neighbours is the goal state, it should just be returned as a
    # normal neighbour in this function

    x,y = current_cell
    neighbours = []
    
    for item in all_cells:
        nX, nY = item[0], item[1]
        if (x == nX + cellSize and y == nY) or (x == nX - cellSize and y ==nY) or (x == nX and y == nY + cellSize) or (x == nX and y == nY - cellSize):
            if item not in obstacles:
                neighbours.append(item)

    return neighbours

pygame.init()
pygame.font.init()
clock = pygame.time.Clock()
    
res, screen, colors, width, height = make_game_basics((900, 800), "Roboto-maze")

'''
Tinyfont is used for small text with corbet font at size 25
small font is used for medium text with corbet font at size 35
bigfont is used for large text with corbet font at size 50
'''
tinyfont = pygame.font.SysFont("Corbel",25)
smallfont = pygame.font.SysFont('Corbel',35) 
bigfont = pygame.font.SysFont("Corbel", 50)

quit_game, quit_game_size = make_text(smallfont, "Quit Game", colors["white"])
#task1, part 2e: changed the text font-family for new game and win game to be tinyfont and bigfont respectively instead of smallfont and tinyfont. 
new_game, new_game_size = make_text(tinyfont, "New Game", colors["white"])
win_game, win_game_size = make_text(bigfont, "Brilliant!", colors["white"])
#task 1, part 2d: changed the win text to 'brilliant' from 'fantastic'
quit_button_coords = [300, 750, 200, 50]
new_game_button_coords = [500, 750, 200, 50]

cellSize = 75

def main():

    flag = False
    click_robot = False
    grid_coords = gridCoords(cellSize)
    all_coords = grid_coords.copy()
    path = []
    robot_start, goal_cell, obstacle_cells = sample_cells(grid_coords, 35) 
        #task 1 part 2b: changed the number of obstacles from 25 to 35
    path.append(robot_start)
    
    while True:
        
        screen.fill(colors["white"]) 
        mouse = pygame.mouse.get_pos() 

        drawGrid(screen, colors["black"], grid_coords, cellSize)
        drawRobot(screen, colors["yellow"], robot_start, cellSize)
        drawGoal(screen, colors["red"], goal_cell, cellSize) 
        #task 1, part 2c: made the goal cell red instead of red to make it more visible and distinct from the robot cell

        drawObstacles(screen, colors["blue"], obstacle_cells, cellSize)

        '''
        The following code block creates two buttons on the screen, 
        one for quitting the game and another for starting a new game.
        They call functions quit_game and new_game respectively when clicked.
        '''
        make_rect_with_text(screen, colors["gray"], quit_button_coords[0], 
                        quit_button_coords[1], quit_button_coords[2], 
                        quit_button_coords[3], 
                        quit_game, quit_button_coords[0]+(quit_game_size[0]/2)-50, 
                       quit_button_coords[1] + (quit_game_size[1]/2)-5)
    
        make_rect_with_text(screen, colors["gray"], new_game_button_coords[0], 
                        new_game_button_coords[1], new_game_button_coords[2], 
                        new_game_button_coords[3], 
                        new_game, new_game_button_coords[0]+(new_game_size[0]/2)-50, 
                       new_game_button_coords[1] + (new_game_size[1]/2)-5)
        
        for ev in pygame.event.get():
            
            if ev.type == pygame.QUIT: 
                pygame.quit() 
                sys.exit()

            #checks if a mouse is clicked 
            '''
            if a mouse is cloked, it checks if the click is within the bounds of the quit button or the new game button.
            0 represents the x coordinate of the mouse click and 1 represents the y coordinate of the mouse click.
            If the click is in the bounds of the quit button, it quits the game.
            If the click is in the bounds of the new game button, starts a new game
            If the click is not within the bounds of either button, check if the move is legal and updates the robot's position.
            It also checks if the goal has been reached by calling the reachedGoal() function and sets a flag to true if the goal is reached
            '''
            if ev.type == pygame.MOUSEBUTTONDOWN: 
               
                if quit_button_coords[0] <= mouse[0] <= quit_button_coords[0]+200 and quit_button_coords[1] <= mouse[1] <= quit_button_coords[1]+50: 
                    pygame.quit() 
                    sys.exit()
    
                if new_game_button_coords[0] <= mouse[0] <= new_game_button_coords[0]+200 and new_game_button_coords[1] <= mouse[1] <= new_game_button_coords[1]+50:
                    main()

                robot_start = legalMove(robot_start, mouse, all_coords, obstacle_cells, width, height, cellSize)
                path.append(robot_start)
                
                if reachedGoal(path, mouse, goal_cell, width, height, cellSize):
                    
                    flag = True
        ''' 
        checks if the robot reached the flag and if so, it displays a message saying fancstic and a button to start a new game.
        '''
        if flag == True:
            make_rect_with_text(screen, colors["gray"], 150-win_game_size[0]/2, 120-win_game_size[1]/2, win_game_size[0], win_game_size[1]+60, 
                        win_game, 150-(win_game_size[0]/2), 
                       120 + (win_game_size[1]/2))


        pygame.display.update()
        clock.tick(60)
main()

SystemExit: 

>**Task 1: (20/100 Points)**
>
> 1. [10 Points] Read through the given code, understand what is going on and add explanatory comments in all indicated functions and blocks of code. Your comments should provide a *brief yet concise and precise* explanation of the functionality of the function/block of code and, in the case of functions, what are the inputs and outputs. An example has been done for you for the function `make_game_basics`. You should finish all other indicated locations in the code with your own comments (there are 10 indicated locations in total, each is awarded with 1 Point). Notice, there are a few comments already in the code to help you orient yourselves.
> 2. [10 Points] Make 5 changes in the provided code in order to customize it (each change receives 2 Points). You need to change the following:
>    - The starting cell of the robot
>    - The number of obstacles used in the game
>    - The colour of the goal cell (by adding a new one and not by using one of the already existing in the code)
>    - The congratulatory message that appears when a player has reached the goal cell
>    - The font of the New Game and Quit Game buttons

## Building an AI to find the solution to the game

Now we need to make an agent that, if we wished, could play the game for us. We will start by considering the game as a search problem and a case where we wish to apply a search algorithm in order to find solutions. 

Your next task is to implement search algorithms that would help your AI explore the search space, explore the available moves and from that, find a path (sequence of moves) that will lead to the goal. 

> **Task 2 (25/100 Points):**
> 
> 1. [15 Points] Write a function **bfs()**, that implements the Breadth First search algorithm. Given a world where each cell is described by the coordinates of its top left corner, in two-dimensions (like in the RoBoToMaZe), the algorithm should receive as input the coordinates of the cells. Some of the coordinates would constitute the starting location of the robot, some the goal, others obstacles and free cells. Then, the function should explore the cells in the system to find a path from the starting cell to the goal cell, following legal moves and the BFS strategy. Your implementation should be such that, upon "generating" the goal cell, the algorithm stops exploring and returns the visited structure, a structure that contains the order in which the legal cells were visited by the algorithm. In order to implement the algorithm, you **should** use the code from notebook uninformed_search_exercises as inspiration and build on that. The grading will be based on the logic of the implementation. Does it use the right data structure? Does it do what it is meant to do, that is, does it visit the cells in the order intended by the algorithm? Does it return the requested and expected result?
> 2. [10 Points] Next step would be to test your function. Before using the RoBoToMaZe for this, we will apply a simpler test on simpler, and more importantly, smaller, two-dimensional worlds. In the cell below, you are given two such scenarios, each implemented using a different approach, both kept simple so that the code can be read through by all programming levels. Test your bfs() implemention using those scenarios and add two more scenarios of your own design. Your scenarios need to challenge the algorithm in some way. Write a few sentences as a comment on what is special in your scenarios - that is, in what way do they try to challenge the algorithm. It is very important that, at this stage of your project, the bfs() implementation works as expected for all the test scenarios! The grading will be based on applying the test correctly, on providing two more scenarios and sufficiently commenting on your scenarios' designs.

In [26]:

from collections import deque


def bfs(starting_cell, goal_cell, legal_moves):

    # Write the function here - remember to add all necessary paremeters in the function declaration above
    visited = []
    queue = [starting_cell]
    while queue:
        current_cell = queue.pop(0)
        if current_cell not in visited:
            visited.append(current_cell)
            if current_cell== goal_cell:
                return visited
            for neighbour in legal_moves[current_cell]:
                if neighbour not in visited and neighbour not in queue:
                    queue.append(neighbour)
    

''' First test scenario : a 4x4 grid world'''
starting_cell = (3,0)
goal_cell = (1,3)

legal_moves = {
    (0,0) : [(0,1), (1,0)],
    (0,1) : [(0,0), (0,2), (1,1)],
    (0,2) : [(0,1), (0,3), (1,2)],
    (0,3) : [(0,2), (1,3)],
    (1,0) : [(0,0), (2,0), (1,1)],
    (1,1) : [(0,1), (1,0), (1,2), (2,1)],
    (1,2) : [(1,1), (0,2), (1,3), (2,2)],
    (1,3) : [(1,2), (0,3), (2,3)],
    (2,0) : [(1,0), (2,1), (3,0)],
    (2,1) : [(2,0), (1,1), (2,2), (3,1)],
    (2,2) : [(1,2), (2,1), (3,2), (2,3)],
    (2,3) : [(2,2), (1,3), (3,3)],
    (3,0) : [(2,0), (3,1)],
    (3,1) : [(3,0), (2,1), (3,2)],
    (3,2) : [(3,1), (2,2), (3,3)],
    (3,3) : [(3,2), (2,3)]
}

## Run your test for the first test scenario here
print(bfs(starting_cell, goal_cell, legal_moves))

''' Second test scenario : a 4x4 grid world with obstacles'''
starting_cell = (3,0)
goal_cell = (1,3)
obstacles = [(1,1), (1,2), (3,1), (3,2)]

def generate_legal_moves(obstacles):

    legal_moves = {}

    for i in range(4):
        for j in range(4):
            
            cur_cell = (i,j)
            all_neighbours = [(i-1,j), (i, j-1),(1+1, j), (i, j+1)]
            legal_neighbours = []
            for item in all_neighbours:
                if item not in obstacles and item[0] >= 0 and item[0]< 4 and item[1] >= 0 and item[1]<4:
                    legal_neighbours.append(item)

            legal_moves[cur_cell] = legal_neighbours

    return legal_moves

## Run your test for the second test scenario here
legal_moves = generate_legal_moves(obstacles)
print(bfs(starting_cell, goal_cell, legal_moves))
''' Third test scenario '''

# Write your test scenario here
'''Goal cell is unreachable due to obstacles'''
starting_cell = (3,0)
goal_cell = (1,3)
obstacles = [(1,1), (1,2), (3,1), (3,2), (0,3), (2,3)]
## Run your test for the third test scenario here
legal_moves = generate_legal_moves(obstacles)
print(bfs(starting_cell, goal_cell, legal_moves))
''' Fourth test scenario '''

# Write your test scenario here
'''starting cell is the same as the goal cell'''
starting_cell = (3,0)
goal_cell = (3,0)
obstacles = []
## Run your test for the fourth test scenario here
legal_moves = generate_legal_moves(obstacles)
print(bfs(starting_cell, goal_cell, legal_moves))

[(3, 0), (2, 0), (3, 1), (1, 0), (2, 1), (3, 2), (0, 0), (1, 1), (2, 2), (3, 3), (0, 1), (1, 2), (2, 3), (0, 2), (1, 3)]
[(3, 0), (2, 0), (1, 0), (2, 1), (0, 0), (2, 2), (0, 1), (2, 3), (0, 2), (1, 3)]
None
[(3, 0)]


> **Task 3 (15/100 Points):**
> 
> Using your code from Task 2 for the BFS algorithm as inspiration, write code for implementing the DFS (Depth First Search) algorithm. The function should return a structure containing the cells in the order they were visited by the algorithm.
>
> Afterwards, use the same test scenarios (all four) from Task 2 to test your DFS implementation.
> 
> Grading: 10 Points are awarded for the logic of the function's implementation. Does it use the right data structure? Does it do what it is meant to do? Does it return the right result? 5 Points are awarded for the tests.

In [ ]:
def dfs(starting_cell, goal_cell, legal_moves):
    stack = [starting_cell]
    visited = []

    while stack:
        current_cell = stack.pop()

        if current_cell in visited:
            continue

        visited.append(current_cell)

        if current_cell == goal_cell:
            break

        for neighbor in legal_moves[current_cell]:
            if neighbor not in visited:
                stack.append(neighbor)

    return visited

# Run your tests for the dfs on the four test scenarios here


> **Task 4 (20/Points)**:
>
> Compare the implementations of BFS and DFS and their performance on your test scenarios. For this, you need to answer the following questions:
>
> 1. [5 Points] How many states (i.e., cells) were generated by each algorithm and for each test scenario until the algorithm terminated?
> 2. [5 Points] Explain the results of the previous question (Task 4.1). For that you should compare both algorithms **and** all scenarios.
> 3. [5 Points] How much time, in seconds, did each algorithm require to return the visited structure for each test scenario? (for this task, you might need to update your functions in tasks 2 and 3 in order to measure time).
> 4. [5 Points] Explain the results of the previous question (Task 4.3). 
>
> How to measure time. You can use the following structure in order to measure the time it takes for the execution of some block of code.
>
> `start_time = time.time()`
> 
> `block of code here`
>
> `end_time = time.time() - start_time`
>
> where `end_time` gives you the desired result.

In [ ]:
# Write your responses for Task 4 here


> **Task 5 (15/100 Points):**
>
> Your next task is to make whatever changes are necessary to your earlier implementations of the BFS algorithm (Task 2) and DFS algorithm (Task 3) so that they can work for the RoBoToMaZe. That is, the updated functions need to be able to receive as input the coordinates of the cells, which of them are obstacles, which are free, where does the robot start from and where does it need to go, and use the corresponding search algorithm's logic to explore the grid in order to find the goal cell.The functions must return a structure `visited` that contains the cells of the grid in the order they were visited by each algorithm.
>
> Call both functions from inside the main function of the game. It should be possible that, upon running the game, first the BFS implementation and then the DFS implementation run and the user is given (in a printed message) the order in which either algorithm visits the cells.
>
> Compare the two algorithms in terms of (a) the number of states generated by each of them and (b) the time it took to run.
> 
> Grading: 5 Points are awarded for giving functions that can work with the game. 5 Points are awarded for running the functions inside the main and at the right location inside the game. Another 5 Points are awarded for correctly comparing the two algorithms runs inside the game in terms of states generated and running time.

In [ ]:
# The updated versions of the algorithms, for Task 5, as well as the comparison details, should be given here


> **Task 6 (5 Points)** : In a concise yet precise and detailed manner, describe how would you implement a function that, given the visited structure containing the cells in the order they were visited by the search algorithm (BFS or DFS) from the starting cell to the goal cell, one could use that to get the solution path.

In [ ]:
# Write your response to Task 6 here

> **BONUS Task (15 Points)** : Give the function **get_path** that implements your logic from Task 6 - that is, given RoBoToMaZe world and the visited structure returned by your BFS() implementation, the **get_path** function should return the solution to follow in order to get from the starting cell to the goal one. Your function should be compatible with the rest of the code for the given and tested (i.e., used) inside the game. Upon running the function, the user should receive a printed message with the solution path to the goal cell, based on the BFS search algorithm.
>
> Responding to this Task is ***OPTIONAL***. A correct implementation can receive max 15 Points, in case you have lost them in other Tasks. If you choose to respond to the Bonus Task and your implementation is not correct, you earn nothing and you lose nothing. 

In [ ]:
# Your code for BONUS Task (OPTIONAL) should go here